# Cross-Entropy, KL Divergence and Softmax

This notebook builds the loss function used by essentially every classifier in machine learning, and shows that each of its pieces is a logarithm doing its usual job. **Cross-entropy** scores a predicted distribution by the average surprise it assigns to what actually happens; the **Kullback–Leibler divergence** is the extra cost of believing the wrong distribution; and **softmax** turns arbitrary scores into probabilities by exponentiating them — so that the logarithm can undo it again.

This is notebook 04 of module 05 (**Information and probability**) in the **Learning Logarithms** series.

## 1. Introduction

An image classifier looks at a photo of a dog and outputs

$$
q = (\text{cat } 0.66,\; \text{dog } 0.24,\; \text{bird } 0.10).
$$

It is wrong — but *how* wrong? A second classifier says (cat $0.40$, dog $0.35$, bird $0.25$). Both pick "cat", so accuracy cannot tell them apart, yet the second is clearly less wrong. And to **train** a classifier we need a score that changes smoothly when the probabilities change, so that calculus can tell us which way to adjust.

Notebook 02 already has the answer. If the truth is "dog", the classifier's **surprise** at the truth is $-\ln q_{\text{dog}}$:

- first classifier: $-\ln 0.24 \approx 1.43$ nats,
- second classifier: $-\ln 0.35 \approx 1.05$ nats,
- a perfect classifier ($q_{\text{dog}} = 1$): $-\ln 1 = 0$.

Averaging this surprise over many examples gives the **cross-entropy**; subtracting the unavoidable surprise (the entropy of the truth) gives the **KL divergence**, a measure of how far one distribution is from another. And because a neural network produces arbitrary real scores, not probabilities, we need a map from scores to probabilities whose logarithm is simple: the **softmax**, $q_i \propto e^{z_i}$, with $\ln q_i = z_i - \ln \sum_j e^{z_j}$.

**Prerequisites**

- Surprise and entropy — [Entropy and information](./02_entropy_and_information.ipynb).
- Log-likelihood and maximum likelihood — [Log-likelihood and MLE](./03_log_likelihood_and_mle.ipynb).
- The `logsumexp` trick and floating-point overflow — [Floating point and numerical stability](../03_computing_logarithms/04_floating_point_and_numerical_stability.ipynb).
- Softmax and cross-entropy in a neural network — [Softmax and cross-entropy](../../../LearningNeuralNetworks/prerequisites/07_softmax_and_cross_entropy.ipynb) (LearningNeuralNetworks).
- Partial derivatives and the chain rule — [Chain rule and computation graphs](../../../LearningNeuralNetworks/prerequisites/04_chain_rule_and_computation_graphs.ipynb) (LearningNeuralNetworks).

After completing this notebook you will be able to:

- Compute the cross-entropy $H(p, q)$ and the KL divergence $D_{\mathrm{KL}}(p \,\|\, q)$ and interpret them in bits as coding costs.
- Prove (and check numerically) Gibbs' inequality $D_{\mathrm{KL}}(p \,\|\, q) \ge 0$, and show that KL is not symmetric.
- Explain why minimizing cross-entropy is the same as maximum likelihood.
- Implement softmax and log-softmax without overflow using `logsumexp`.
- Derive the gradient of the cross-entropy loss with respect to the logits: $q - p$.

## 2. Theory

### 2.1 Cross-entropy: the surprise of the truth under the prediction

Let $p$ be the true distribution over classes and $q$ a predicted one. The **cross-entropy** is the expected surprise, measured with $q$, of outcomes drawn from $p$:

$$
H(p, q) = E_{X \sim p}\left[-\ln q(X)\right] = -\sum_i p_i \ln q_i .
$$

(With $\log_2$ it is in bits, with $\ln$ in nats — machine-learning code uses $\ln$.) When $p$ is the one-hot vector of the correct class $y$, the sum has a single term: $H(p, q) = -\ln q_y$, exactly the surprise of the Introduction.

**Coding interpretation.** By notebook 02 the ideal codeword length for symbol $i$ is $-\log_2 q_i$ bits if you *believe* the distribution is $q$. If the symbols really come from $p$, your average codeword length is $H(p, q)$ bits — and the best possible average, using the true $p$, is $H(p)$.

### 2.2 KL divergence: the price of the wrong belief

The **Kullback–Leibler divergence** is the extra cost:

$$
D_{\mathrm{KL}}(p \,\|\, q) = H(p, q) - H(p) = \sum_i p_i \ln \frac{p_i}{q_i}.
$$

| Exponent form | Log form |
|---|---|
| $\dfrac{p_i}{q_i}$, the likelihood ratio of truth to model | $\ln \dfrac{p_i}{q_i} = \ln p_i - \ln q_i$, averaged over $p$ |

**Gibbs' inequality:** $D_{\mathrm{KL}}(p \,\|\, q) \ge 0$, with equality only when $q = p$. Proof with $\ln x \le x - 1$ (summing over $i$ with $p_i > 0$):

$$
-D_{\mathrm{KL}}(p \,\|\, q) = \sum_i p_i \ln \frac{q_i}{p_i} \le \sum_i p_i \left(\frac{q_i}{p_i} - 1\right) = \sum_i q_i - \sum_i p_i \le 1 - 1 = 0 .
$$

So $H(p, q) \ge H(p)$: no wrong belief ever compresses the truth better than the truth itself. Two warnings:

- **Not symmetric:** in general $D_{\mathrm{KL}}(p \,\|\, q) \neq D_{\mathrm{KL}}(q \,\|\, p)$, so it is a "divergence", not a distance.
- **Can be infinite:** if $q_i = 0$ for an outcome with $p_i > 0$, then $-\ln q_i = \infty$. A model that declares something impossible pays an infinite penalty when it happens.

### 2.3 Minimizing cross-entropy is maximum likelihood

For training data $y_1, \dots, y_n$, let $p$ be the empirical distribution (each observed label gets weight $1/n$). Then

$$
H(p, q_\theta) = -\frac1n \sum_{k=1}^n \ln q_\theta(y_k) = -\frac{1}{n}\,\ell(\theta),
$$

minus the average log-likelihood of notebook 03. Since $H(p)$ does not depend on $\theta$, minimizing cross-entropy, minimizing KL divergence and maximizing likelihood are **the same optimization**.

### 2.4 Softmax: exponentiate, then normalize

A network outputs real scores $z_1, \dots, z_K$ (**logits**). To make probabilities they must be positive and sum to $1$:

$$
q_i = \operatorname{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}.
$$

The exponential makes every score positive and turns score **differences** into probability **ratios**: $q_i / q_j = e^{z_i - z_j}$. Read with logarithms, the logits **are** log-probabilities, up to one shared constant:

$$
\ln q_i = z_i - \operatorname{LSE}(z), \qquad \operatorname{LSE}(z) = \ln \sum_{j} e^{z_j}.
$$

Adding the same constant $c$ to every logit changes nothing ($e^{z_i + c} / \sum e^{z_j + c} = e^{z_i} / \sum e^{z_j}$). That shift invariance is the numerical cure for overflow: with $m = \max_j z_j$,

$$
\operatorname{LSE}(z) = m + \ln \sum_j e^{z_j - m},
$$

where every exponent is $\le 0$, so nothing overflows and at least one term equals $1$, so nothing underflows to a $\ln 0$.

### 2.5 The gradient: prediction minus truth

Combine softmax and cross-entropy. With $\sum_i p_i = 1$,

$$
\mathcal L(z) = -\sum_i p_i \ln q_i = -\sum_i p_i \bigl(z_i - \operatorname{LSE}(z)\bigr) = \operatorname{LSE}(z) - \sum_i p_i z_i .
$$

The logarithm of the softmax has removed every exponential except inside $\operatorname{LSE}$, whose derivative is the softmax itself: $\frac{\partial}{\partial z_k} \ln \sum_j e^{z_j} = \frac{e^{z_k}}{\sum_j e^{z_j}} = q_k$. Hence

$$
\frac{\partial \mathcal L}{\partial z_k} = q_k - p_k .
$$

The gradient is simply **prediction minus truth** — bounded between $-1$ and $1$, never vanishing while the prediction is wrong. This clean form is why softmax and cross-entropy are always paired.

## 3. Worked Examples

### Example 1 — scoring the two classifiers

Truth: dog. Losses $-\ln q_{\text{dog}}$: first classifier $-\ln 0.24 \approx 1.427$, second $-\ln 0.35 \approx 1.050$ nats. The second is better by $0.377$ nats, i.e. its probability on the truth is $e^{0.377} \approx 1.46$ times larger.

### Example 2 — cross-entropy and KL in bits

$p = (\tfrac12, \tfrac14, \tfrac14)$, $q = (0.8, 0.1, 0.1)$.

- $H(p) = \tfrac12 \cdot 1 + \tfrac14 \cdot 2 + \tfrac14 \cdot 2 = 1.5$ bits.
- $H(p, q) = -\tfrac12 \log_2 0.8 - \tfrac14 \log_2 0.1 - \tfrac14 \log_2 0.1 \approx 0.161 + 0.830 + 0.830 = 1.822$ bits.
- $D_{\mathrm{KL}}(p \,\|\, q) = 1.822 - 1.5 = 0.322$ bits: coding $p$-data with a $q$-code wastes about a third of a bit per symbol.
- The other direction: $D_{\mathrm{KL}}(q \,\|\, p) = 0.8 \log_2 1.6 + 2 \times 0.1 \log_2 0.4 \approx 0.542 - 0.264 = 0.278$ bits. Different.

### Example 3 — softmax by hand

Logits $z = (2.0, 1.0, 0.1)$. $e^{z} \approx (7.389, 2.718, 1.105)$, sum $11.212$. So $q \approx (0.659, 0.242, 0.099)$ — the first classifier of the Introduction. $\operatorname{LSE}(z) = \ln 11.212 \approx 2.417$, and $\ln q_2 = 1.0 - 2.417 = -1.417$, i.e. the loss for "dog" is $1.417$ nats.

### Example 4 — the gradient

Same logits, truth "dog": $p = (0, 1, 0)$. Gradient $q - p \approx (0.659, -0.758, 0.099)$. Gradient descent *lowers* the cat and bird logits and *raises* the dog logit, in proportion to how wrong each probability is.

### Example 5 — overflow and the shift

Logits $(1000, 999, 998)$: $e^{1000}$ overflows a double (the limit is about $e^{709.8}$). Subtract the maximum: $(0, -1, -2)$, $e^{(\cdot)} = (1, 0.368, 0.135)$, sum $1.503$, so $q \approx (0.665, 0.245, 0.090)$ and $\operatorname{LSE} = 1000 + \ln 1.503 \approx 1000.408$.

## 4. Python Examples

### 4.1 Entropy, cross-entropy and KL

In [ ]:
import numpy as np


def entropy(p, base=np.e):
    p = np.asarray(p, dtype=float)
    mask = p > 0
    return float(-np.sum(p[mask] * np.log(p[mask])) / np.log(base))


def cross_entropy(p, q, base=np.e):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    mask = p > 0
    with np.errstate(divide="ignore"):
        return float(-np.sum(p[mask] * np.log(q[mask])) / np.log(base))


def kl_divergence(p, q, base=np.e):
    return cross_entropy(p, q, base) - entropy(p, base)


p = [0.5, 0.25, 0.25]
q = [0.8, 0.1, 0.1]
print(f"H(p)       = {entropy(p, 2):.4f} bits")
print(f"H(p, q)    = {cross_entropy(p, q, 2):.4f} bits")
print(f"KL(p || q) = {kl_divergence(p, q, 2):.4f} bits")
print(f"KL(q || p) = {kl_divergence(q, p, 2):.4f} bits   <- not symmetric")
print(f"KL(p || p) = {kl_divergence(p, p, 2):.4f} bits")
print("KL with an impossible outcome:", kl_divergence([0.5, 0.5], [1.0, 0.0]))

### 4.2 Gibbs' inequality on random distributions

Draw $100{,}000$ random pairs $(p, q)$ on five outcomes; the KL divergence is never negative, and it is $0$ only when the two coincide.

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
p = rng.dirichlet(np.ones(5), size=100_000)
q = rng.dirichlet(np.ones(5), size=100_000)
kl_values = np.sum(p * np.log(p / q), axis=1)
reverse_kl_values = np.sum(q * np.log(q / p), axis=1)

print("smallest KL(p || q) seen :", kl_values.min())
print("any negative?            :", bool(np.any(kl_values < -1e-12)))
print("KL(p||q) == KL(q||p) for :", int(np.sum(np.isclose(kl_values, reverse_kl_values))), "of", len(kl_values), "pairs")
print("KL(p || p) for 3 samples :", np.sum(p[:3] * np.log(p[:3] / p[:3]), axis=1))

### 4.3 Softmax, naive and stable

The naive formula overflows for large logits and returns `nan`. The stable version subtracts the maximum first; `np.logaddexp.reduce` computes $\operatorname{LSE}$ the same way.

In [ ]:
import numpy as np


def softmax_naive(z):
    exponentials = np.exp(z)
    return exponentials / exponentials.sum()


def logsumexp(z):
    z = np.asarray(z, dtype=float)
    largest = z.max()
    return largest + np.log(np.sum(np.exp(z - largest)))


def log_softmax(z):
    z = np.asarray(z, dtype=float)
    return z - logsumexp(z)


def softmax(z):
    return np.exp(log_softmax(z))


small_logits = np.array([2.0, 1.0, 0.1])
large_logits = np.array([1000.0, 999.0, 998.0])
print("softmax(2, 1, 0.1)        :", np.round(softmax(small_logits), 4), " naive:", np.round(softmax_naive(small_logits), 4))
with np.errstate(over="ignore", invalid="ignore"):
    print("naive softmax(1000,999,998):", softmax_naive(large_logits))
print("stable softmax(1000,...)   :", np.round(softmax(large_logits), 4))
print("logsumexp(1000, 999, 998)  :", round(logsumexp(large_logits), 4), " numpy:", round(float(np.logaddexp.reduce(large_logits)), 4))
print("log_softmax(2, 1, 0.1)     :", np.round(log_softmax(small_logits), 4))

### 4.4 Cross-entropy loss from logits, and its gradient

Computing $-\ln(\operatorname{softmax}(z)_y)$ in two steps can fail: if the softmax underflows to $0$, the log is $-\infty$. Computing it as $\operatorname{LSE}(z) - z_y$ never fails. We then check the formula $\nabla_z \mathcal L = q - p$ against finite differences.

In [ ]:
import numpy as np


def logsumexp(z):
    largest = np.max(z)
    return largest + np.log(np.sum(np.exp(z - largest)))


def loss_from_logits(z, true_class):
    return logsumexp(z) - z[true_class]


def gradient_from_logits(z, true_class):
    q = np.exp(z - logsumexp(z))
    p = np.zeros_like(z)
    p[true_class] = 1.0
    return q - p


z = np.array([2.0, 1.0, 0.1])
true_class = 1
print("loss                 :", round(loss_from_logits(z, true_class), 4), "nats")
analytic = gradient_from_logits(z, true_class)
step = 1e-6
numeric = np.array([(loss_from_logits(z + step * e, true_class) - loss_from_logits(z - step * e, true_class)) / (2 * step) for e in np.eye(3)])
print("gradient q - p       :", np.round(analytic, 6))
print("finite differences   :", np.round(numeric, 6))
print("agree:", np.allclose(analytic, numeric, atol=1e-8))

extreme = np.array([0.0, -800.0, 5.0])
with np.errstate(divide="ignore"):
    two_step = -np.log(np.exp(extreme - logsumexp(extreme))[1])
print("two-step loss for a hopeless logit:", two_step, "   stable loss:", loss_from_logits(extreme, 1))

### 4.5 Training a tiny softmax classifier with the $q - p$ gradient

Three clusters of points in the plane, a linear model $z = W x + b$, and gradient descent using only the rule "gradient of the loss with respect to the logits is $q - p$" (the chain rule then multiplies by $x$ for $W$). The average cross-entropy falls towards its floor.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
centres = np.array([[0.0, 2.0], [2.0, -1.0], [-2.0, -1.0]])
labels = np.repeat(np.arange(3), 100)
points = centres[labels] + rng.normal(scale=0.9, size=(300, 2))
one_hot = np.eye(3)[labels]

weights = np.zeros((2, 3))
bias = np.zeros(3)
learning_rate = 0.5
for epoch in range(201):
    logits = points @ weights + bias
    log_normalizer = np.logaddexp.reduce(logits, axis=1, keepdims=True)
    log_q = logits - log_normalizer
    loss = -np.mean(np.sum(one_hot * log_q, axis=1))
    gradient_logits = (np.exp(log_q) - one_hot) / len(points)     # q - p, averaged
    weights -= learning_rate * points.T @ gradient_logits
    bias -= learning_rate * gradient_logits.sum(axis=0)
    if epoch in (0, 10, 50, 200):
        accuracy = np.mean(np.argmax(logits, axis=1) == labels)
        print(f"epoch {epoch:>3}: cross-entropy {loss:.4f} nats, accuracy {accuracy:.3f}")
print(f"(a model that always predicts uniform scores ln 3 = {np.log(3):.4f} nats)")

At epoch $0$ every logit is $0$, the prediction is uniform, and the loss is exactly $\ln 3$ — the entropy of a uniform guess over three classes.

## 5. Visualizations

### 5.1 The loss $-\ln q$ for the true class

Small mistakes are cheap, confident mistakes are very expensive. On a log-scaled $q$ axis the loss is a straight line: each factor of $10$ less probability on the truth costs another $\ln 10 \approx 2.3$ nats.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

q = np.logspace(-4, 0, 400)
fig, (ax_linear, ax_log) = plt.subplots(1, 2, figsize=(12, 4.5))
ax_linear.plot(q, -np.log(q), color="tab:red")
for probability, name in [(0.24, "classifier 1"), (0.35, "classifier 2")]:
    ax_linear.plot(probability, -np.log(probability), "o", color="black")
    ax_linear.annotate(f"{name}: {-np.log(probability):.2f}", (probability, -np.log(probability)), textcoords="offset points", xytext=(10, 10))
ax_linear.set_xlabel("predicted probability of the true class q (linear scale)")
ax_linear.set_ylabel("loss -ln q (nats)")
ax_linear.set_title("Cross-entropy loss of one example")
ax_linear.set_ylim(0, 6)
ax_linear.grid(True, alpha=0.3)

ax_log.plot(q, -np.log(q), color="tab:red")
ax_log.set_xscale("log")
ax_log.set_xlabel("predicted probability of the true class q (log scale)")
ax_log.set_ylabel("loss -ln q (nats)")
ax_log.set_title("Straight line: each factor 10 costs ln 10 = 2.30 nats")
ax_log.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 KL is not symmetric

Fix the true coin $p = 0.2$ and vary the model $q$. Both divergences are $0$ at $q = 0.2$, but $D_{\mathrm{KL}}(p \,\|\, q)$ grows without bound as $q \to 0$ or $q \to 1$, because the model then calls a real outcome nearly impossible. It grows slowly near $q = 0$ (the outcome being ruled out, heads, happens only $20\%$ of the time) and steeply near $q = 1$ (tails, ruled out there, happens $80\%$ of the time). Meanwhile $D_{\mathrm{KL}}(q \,\|\, p)$ stays finite at both ends: at $q = 0$ it equals $-\ln 0.8 \approx 0.22$ and at $q = 1$ it equals $-\ln 0.2 \approx 1.61$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

p_true = 0.2
q = np.linspace(0.001, 0.999, 999)
forward = p_true * np.log(p_true / q) + (1 - p_true) * np.log((1 - p_true) / (1 - q))
reverse = q * np.log(q / p_true) + (1 - q) * np.log((1 - q) / (1 - p_true))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(q, forward, color="tab:blue", label="KL(p || q)")
ax.plot(q, reverse, color="tab:orange", label="KL(q || p)")
ax.axvline(p_true, color="gray", linestyle=":", label="q = p = 0.2")
ax.set_xlabel("model probability q of heads")
ax.set_ylabel("divergence (nats)")
ax.set_ylim(0, 3)
ax.set_title("Two directions of KL for a coin with p = 0.2")
ax.legend(loc="upper center")
ax.grid(True, alpha=0.3)
plt.show()

### 5.3 Cross-entropy = entropy + KL

For the distributions of Example 2 and two others, each bar splits $H(p, q)$ into the unavoidable part $H(p)$ and the penalty $D_{\mathrm{KL}}(p \,\|\, q)$ for the wrong belief.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

p = np.array([0.5, 0.25, 0.25])
models = {
    "q = p": p,
    "q = (0.4, 0.3, 0.3)": np.array([0.4, 0.3, 0.3]),
    "q = uniform": np.full(3, 1 / 3),
    "q = (0.8, 0.1, 0.1)": np.array([0.8, 0.1, 0.1]),
    "q = (0.1, 0.1, 0.8)": np.array([0.1, 0.1, 0.8]),
}
entropy_bits = -np.sum(p * np.log2(p))
names = list(models)
kl_bits = [np.sum(p * np.log2(p / q)) for q in models.values()]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(names, [entropy_bits] * len(names), color="tab:gray", label="H(p) = 1.5 bits (unavoidable)")
ax.bar(names, kl_bits, bottom=entropy_bits, color="tab:red", alpha=0.75, label="KL(p || q) (cost of the wrong belief)")
for x, kl in enumerate(kl_bits):
    ax.text(x, entropy_bits + kl + 0.03, f"H(p,q) = {entropy_bits + kl:.3f}", ha="center", fontsize=9)
ax.set_ylabel("bits per symbol")
ax.set_ylim(0, 3.1)
ax.set_title("Cross-entropy H(p, q) for p = (0.5, 0.25, 0.25)")
ax.legend(loc="upper left")
ax.grid(True, axis="y", alpha=0.3)
plt.show()

### 5.4 Softmax with a temperature

Dividing the logits by a temperature $T$ before the softmax, $q = \operatorname{softmax}(z / T)$, sharpens ($T < 1$) or flattens ($T > 1$) the distribution. On the log scale this is literal: $\ln q_i = z_i / T - \operatorname{LSE}(z / T)$, so $T$ just rescales the log-probability differences. The right panel shows the entropy of the softmax rising from $0$ (one-hot, $T \to 0$) to $\ln 3$ (uniform, $T \to \infty$).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

logits = np.array([2.0, 1.0, 0.1])
temperatures = np.logspace(-1.5, 1.5, 300)
probabilities = []
for temperature in temperatures:
    scaled = logits / temperature
    probabilities.append(np.exp(scaled - np.logaddexp.reduce(scaled)))
probabilities = np.array(probabilities)
entropies = -np.sum(probabilities * np.log(np.clip(probabilities, 1e-300, None)), axis=1)

fig, (ax_probabilities, ax_entropy) = plt.subplots(1, 2, figsize=(12, 4.5))
for index, name in enumerate(["cat (z = 2.0)", "dog (z = 1.0)", "bird (z = 0.1)"]):
    ax_probabilities.plot(temperatures, probabilities[:, index], label=name)
ax_probabilities.set_xscale("log")
ax_probabilities.set_xlabel("temperature T (log scale)")
ax_probabilities.set_ylabel("softmax(z / T)")
ax_probabilities.set_title("Low T: winner takes all; high T: uniform")
ax_probabilities.grid(True, which="both", alpha=0.3)
ax_probabilities.legend(loc="center right")

ax_entropy.plot(temperatures, entropies, color="black")
ax_entropy.axhline(np.log(3), color="crimson", linestyle="--", label="ln 3 (uniform)")
ax_entropy.set_xscale("log")
ax_entropy.set_xlabel("temperature T (log scale)")
ax_entropy.set_ylabel("entropy of softmax (nats)")
ax_entropy.set_title("Entropy grows with temperature")
ax_entropy.grid(True, which="both", alpha=0.3)
ax_entropy.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

1. The truth is class A with certainty, $p = (1, 0)$, and a model predicts $q = (0.8, 0.2)$. Compute $H(p, q)$ in bits and in nats.
2. Compute $\operatorname{softmax}(0, \ln 2, \ln 3)$ exactly.
3. Explain why $D_{\mathrm{KL}}(p \,\|\, p) = 0$ directly from the formula.

### Intermediate

4. **Estimate it first, then compute.** Using $\ln 10 \approx 2.3$, estimate the loss in nats when the true class gets probability $0.01$, and the loss of a uniform guess over $1000$ classes. Then compute both.
5. For coins with $p = 0.5$ and $q = 0.9$ (probabilities of heads), compute $D_{\mathrm{KL}}(p \,\|\, q)$ and $D_{\mathrm{KL}}(q \,\|\, p)$ in nats.
6. Show that $\operatorname{LSE}(z + c) = \operatorname{LSE}(z) + c$ and use it to compute $\operatorname{LSE}(1000, 1000)$ exactly. What does `np.log(np.sum(np.exp([1000.0, 1000.0])))` return?

### Advanced

7. Let $\mathcal L(z) = -\sum_i p_i \ln \operatorname{softmax}(z)_i$ for three classes. Use SymPy to differentiate with respect to $z_1, z_2, z_3$ and verify that the gradient equals $q - p$ (with $p_1 + p_2 + p_3 = 1$).
8. Show that $\max_i z_i \le \operatorname{LSE}(z) \le \max_i z_i + \ln K$ for $K$ logits. What does this say about $\operatorname{LSE}$ as a "soft maximum"?
9. Let $p$ be the true distribution and suppose we fit $q$ by minimizing $H(p, q)$ over all distributions on the same outcomes. Using Gibbs' inequality, show that the minimizer is $q = p$, and check numerically by minimizing $H(p, \operatorname{softmax}(z))$ over $z$ with gradient descent for $p = (0.5, 0.3, 0.2)$.

## 7. Solutions

1. $H(p, q) = -\log_2 0.8 \approx 0.322$ bits $= -\ln 0.8 \approx 0.223$ nats. (Here $H(p) = 0$, so this is also the KL divergence.)
2. $e^{0}, e^{\ln 2}, e^{\ln 3} = 1, 2, 3$, sum $6$: softmax $= (\tfrac16, \tfrac13, \tfrac12)$. Logits that are logs of numbers give probabilities proportional to those numbers.
3. Every term is $p_i \ln (p_i / p_i) = p_i \ln 1 = 0$.
4. $-\ln 0.01 = 2 \ln 10 \approx 4.6$ nats (exactly $4.605$). Uniform over $1000$: $-\ln(1/1000) = 3 \ln 10 \approx 6.9$ nats (exactly $6.908$) — a useful sanity check: an untrained classifier with $K$ classes should start near $\ln K$.
5. $D_{\mathrm{KL}}(p \,\|\, q) = 0.5 \ln \tfrac{0.5}{0.9} + 0.5 \ln \tfrac{0.5}{0.1} \approx -0.294 + 0.805 = 0.511$ nats. $D_{\mathrm{KL}}(q \,\|\, p) = 0.9 \ln 1.8 + 0.1 \ln 0.2 \approx 0.529 - 0.161 = 0.368$ nats.
6. $\ln \sum e^{z_j + c} = \ln\left(e^c \sum e^{z_j}\right) = c + \operatorname{LSE}(z)$. So $\operatorname{LSE}(1000, 1000) = 1000 + \operatorname{LSE}(0, 0) = 1000 + \ln 2 \approx 1000.693$. The naive expression overflows: `exp(1000.0)` is `inf`, and the result is `inf`.
7. See the SymPy check below: the difference between the derivative and $q_k - p_k$ simplifies to $0$ once $p_3 = 1 - p_1 - p_2$ is substituted.
8. $\sum_j e^{z_j} \ge e^{\max z}$ (one term) and $\le K e^{\max z}$ (each term at most the largest). Taking $\ln$: $\max z \le \operatorname{LSE}(z) \le \max z + \ln K$. So $\operatorname{LSE}$ is a smooth over-estimate of the maximum, off by at most $\ln K$ — hence the name "soft max" for its gradient, the softmax.
9. $H(p, q) = H(p) + D_{\mathrm{KL}}(p \,\|\, q) \ge H(p)$ with equality only at $q = p$. Gradient descent with gradient $\operatorname{softmax}(z) - p$ converges to $q = (0.5, 0.3, 0.2)$ (below), with logits equal to $\ln p_i$ plus a constant.

Numerical checks:

In [ ]:
import math

import numpy as np
import sympy as sp

print("Ex 1:", round(-math.log2(0.8), 4), "bits,", round(-math.log(0.8), 4), "nats")
z = np.array([0.0, math.log(2), math.log(3)])
print("Ex 2:", np.round(np.exp(z) / np.exp(z).sum(), 6))
print("Ex 4:", round(-math.log(0.01), 4), round(math.log(1000), 4))
p_heads, q_heads = 0.5, 0.9
forward = p_heads * math.log(p_heads / q_heads) + (1 - p_heads) * math.log((1 - p_heads) / (1 - q_heads))
reverse = q_heads * math.log(q_heads / p_heads) + (1 - q_heads) * math.log((1 - q_heads) / (1 - p_heads))
print(f"Ex 5: KL(p||q) = {forward:.4f}, KL(q||p) = {reverse:.4f}")
with np.errstate(over="ignore"):
    print("Ex 6: stable", float(np.logaddexp.reduce([1000.0, 1000.0])), " 1000 + ln 2 =", 1000 + math.log(2),
          " naive", np.log(np.sum(np.exp([1000.0, 1000.0]))))

z1, z2, z3, p1, p2 = sp.symbols("z1 z2 z3 p1 p2", real=True)
logits = [z1, z2, z3]
p_vector = [p1, p2, 1 - p1 - p2]
normalizer = sum(sp.exp(zi) for zi in logits)
q_vector = [sp.exp(zi) / normalizer for zi in logits]
loss = -sum(pi * sp.log(qi) for pi, qi in zip(p_vector, q_vector))
differences = [sp.simplify(sp.diff(loss, zk) - (qk - pk)) for zk, qk, pk in zip(logits, q_vector, p_vector)]
print("Ex 7: dL/dz_k - (q_k - p_k) =", differences)

rng = np.random.default_rng(5)
test_logits = rng.normal(size=(5, 4)) * 10
lse = np.logaddexp.reduce(test_logits, axis=1)
print("Ex 8: bounds hold:", bool(np.all(test_logits.max(axis=1) <= lse) and np.all(lse <= test_logits.max(axis=1) + np.log(4))))

target = np.array([0.5, 0.3, 0.2])
logits_fit = np.zeros(3)
for _ in range(2000):
    q = np.exp(logits_fit - np.logaddexp.reduce(logits_fit))
    logits_fit -= 1.0 * (q - target)
q = np.exp(logits_fit - np.logaddexp.reduce(logits_fit))
print("Ex 9: fitted q =", np.round(q, 6), " logits - ln p =", np.round(logits_fit - np.log(target), 6))

## 8. Conclusion

- **Cross-entropy** $H(p, q) = -\sum p_i \ln q_i$ is the average surprise of the truth under the prediction; for a one-hot truth it is $-\ln q_y$, the standard classification loss.
- **KL divergence** $D_{\mathrm{KL}}(p \,\|\, q) = H(p, q) - H(p) = \sum p_i \ln \frac{p_i}{q_i}$ is the extra cost of believing $q$ when the truth is $p$. Gibbs' inequality: it is $\ge 0$, zero only for $q = p$. It is not symmetric and can be infinite.
- Minimizing cross-entropy over a model is maximizing its log-likelihood (notebook 03).
- **Softmax** $q_i = e^{z_i} / \sum_j e^{z_j}$ exponentiates so that the logarithm can undo it: $\;q_i = e^{z_i - \operatorname{LSE}(z)} \iff \ln q_i = z_i - \operatorname{LSE}(z)$. Logits are log-probabilities up to a constant.
- Numerically, always subtract the maximum logit (`logsumexp`, `np.logaddexp.reduce`) and compute the loss as $\operatorname{LSE}(z) - z_y$, never as the log of a computed probability.
- The gradient of softmax cross-entropy with respect to the logits is **$q - p$**: prediction minus truth.
- The picture: on a log-scaled probability axis the loss $-\ln q$ is a straight line — every factor of $10$ less confidence in the truth costs another $2.3$ nats.

**Next:** notebook 05 leaves discrete distributions for continuous ones built from **multiplicative** randomness: the log-normal distribution, and why so many real-world numbers start with the digit $1$ (Benford's law).